# 03 · LLM zero-shot: piloto de modelos

Paso 4.1 del flujo. Se comparan tres LLMs abiertos en **zero-shot sobre dev** con el mismo prompt y la misma entrada limpia, para elegir el modelo con el que seguir (ajuste del prompt, few-shot y RAG).

- **No se genera texto:** una sola pasada por noticia y se lee la probabilidad del primer token de la respuesta. `p_fake = P(Falsa) / (P(Falsa) + P(Verdadera))`. Así hay AUC y se pueden aplicar las mismas reglas de decisión que a los baselines (umbral 0,5 y rango al 50 %).
- **Criterio de elección:** AUC en dev (no depende del umbral); desempate por F1 con rango, `p_formato` (si el modelo responde en el formato pedido) y tiempo.
- **Test no se usa en este notebook.**

## Entorno

Pensado para **Kaggle**: *Accelerator* → GPU T4 ×2, *Internet* activado y un secreto `HF_TOKEN` (*Add-ons → Secrets*). Llama 3.1 y Gemma 3 son modelos restringidos: hay que aceptar su licencia en Hugging Face con la misma cuenta del token. Qwen3 no lo necesita.

La celda clona el repo, instala dependencias y genera `data/processed` si no existe. En local (sin GPU) el notebook se ejecuta igual pero se salta los modelos.

In [10]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/LLM"

if IN_KAGGLE:
    ROOT = Path("/kaggle/working/Practica-TAPLN")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.51", "accelerate", "bitsandbytes"], check=True)
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos (Qwen).", e)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

In [11]:
import numpy as np
import pandas as pd
import torch

import data_utils as du
import preprocessing as pp
import evaluation as ev
import baselines as bl
import llm

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
dev_in = llm.llm_inputs(splits["dev"], media_re)
OUT = ROOT / "results" / "llm"
OUT.mkdir(parents=True, exist_ok=True)

print(f"dev: {len(dev_in)} noticias, {dev_in.label.mean():.1%} Fake")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "ninguna")

dev: 290 noticias, 47.9% Fake
GPUs: ['Tesla T4', 'Tesla T4']


## Prompt

Mismo prompt para los tres modelos (en `src/llm.py`): rol de verificador, aviso de que se han quitado enlaces y medios y de que las cifras aparecen como `NUM`, titular («no disponible» si falta) y cuerpo cortado a 800 palabras. En Qwen3 se desactiva el modo *thinking*; probarlo queda para el ajuste del prompt.

In [12]:
print(llm.SYSTEM, "\n")
print(llm.USER.format(headline=dev_in.headline[0], body=dev_in.body[0][:300] + " [...]"))

Eres un verificador de noticias en español. Lees una noticia y decides si es verdadera o falsa. En el texto se han eliminado enlaces y nombres de medios, y las cifras aparecen como NUM. Responde con una sola palabra: Verdadera o Falsa. 

Titular: MAESTRA DE NUM AÑOS QUE TUVO RELACIONES CON UN ALUMNO DE NUM DARÁ EDUCACIÓN SEXUAL EN EL CONALEP

Cuerpo:
Luego de que se revelará el caso de una maestra estadounidense de NUM años de nombre Alexandria Vera que fue acusada de abusar de uno de sus alumno de NUM años (con tuvo un noviazgo e incluso quedó embarazada), el juez que analizó la situación finalmente determinó que la profesora no será encarcelad [...]

¿Esta noticia es verdadera o falsa? Responde solo «Verdadera» o «Falsa».


## Candidatos

| Clave | Modelo | Carga en T4 | Licencia |
|---|---|---|---|
| `qwen3-8b` | Qwen/Qwen3-8B | fp16, repartido en 2×T4 | Apache 2.0 |
| `llama31-8b` | meta-llama/Llama-3.1-8B-Instruct | fp16, repartido en 2×T4 | Llama 3.1 Community |
| `gemma3-12b` | google/gemma-3-12b-it | 4 bits (NF4) | Gemma Terms |

Gemma va en 4 bits porque en fp16 ocupa ~24 GB de los 32 disponibles. Si alguno da `nan` (desbordamiento de fp16), se anota y se repite con otra configuración.

Cada modelo guarda `results/llm/piloto_dev_<clave>.csv` y, si la sesión se corta, reanuda desde ahí. Con `EJECUTAR` se puede lanzar un solo modelo por sesión.

In [13]:
MODELOS = {
    "qwen3-8b":   {"name": "Qwen/Qwen3-8B"},
    "llama31-8b": {"name": "meta-llama/Llama-3.1-8B-Instruct"},
    "gemma3-12b": {"name": "google/gemma-3-12b-it", "quant": "4bit", "dtype": torch.float32},
} 

EJECUTAR = ["gemma3-12b"]
BORRAR_CACHE = IN_KAGGLE   # borra cada modelo del disco al terminar con él

In [14]:
for key in EJECUTAR:
    cfg, path = MODELOS[key], OUT / f"piloto_dev_{key}.csv"
    if path.exists() and len(pd.read_csv(path)) == len(dev_in):
        print(f"{key}: ya hecho"); continue
    print(f"== {key} ({cfg['name']})")
    tok, model = llm.load_model(cfg["name"], cfg.get("quant"), cfg.get("dtype"))
    print("Tokens de respuesta:", {k: [tok.decode([i]) for i in v] for k, v in llm.answer_ids(tok).items()})
    # Comprobación a mano del formato con tres noticias
    for i in range(3):
        p = llm.render(tok, llm.build_messages(dev_in.headline[i], dev_in.body[i]))
        print(f"  ejemplo {i} (label={dev_in.label[i]}): {llm.generate_answer(model, tok, p)!r}")
    llm.run(model, tok, dev_in, path)
    del model, tok
    llm.release(cfg["name"], delete_cache=BORRAR_CACHE)

== gemma3-12b (google/gemma-3-12b-it)


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Tokens de respuesta: {'fake': [' F', ' falso', ' falsa', ' FAL', 'f', 'F'], 'true': [' Ver', 'VER', 'Ver', ' Verd', ' VER', 'verd', 'verdad', ' verdadero', ' verdadera']}
  ejemplo 0 (label=1): 'Falsa'
  ejemplo 1 (label=0): 'Verdadera'
  ejemplo 2 (label=0): 'Verdadera'
  50/290  (5.21 s/noticia)
  100/290  (5.37 s/noticia)
  150/290  (5.30 s/noticia)
  200/290  (5.35 s/noticia)
  250/290  (5.31 s/noticia)


## Resultados en dev

Se compara con el baseline char-SVM entrenado solo con train (mismo protocolo que en `02_baselines`). Columnas extra: `p_formato` medio (≈1 si el modelo responde «Verdadera»/«Falsa»), número de `nan` y segundos por noticia.

In [15]:
tr, dev = splits["train"], splits["dev"]
rows = []

svm = bl.make_pipe("char", 1.0, "svm").fit(pp.prepare(tr, media_re), tr.label)
s = bl.get_scores(svm, pp.prepare(dev, media_re))
rows.append({**ev.evaluate(dev.label, ev.predict_threshold(s, 0.0), s, "char-svm (baseline)"), "decision": "umbral"})
rows.append({**ev.evaluate(dev.label, ev.predict_by_rank(s, 0.5), s, "char-svm (baseline)"), "decision": "rango 50 %"})

piloto = {}
for key in MODELOS:
    path = OUT / f"piloto_dev_{key}.csv"
    if not path.exists():
        continue
    r = piloto[key] = pd.read_csv(path)
    s, y = r.p_fake.to_numpy(), r.label.to_numpy()
    extra = {"p_formato": r.p_formato.mean(), "nan": int(r.nan.sum()), "s_noticia": r.segundos.mean()}
    rows.append({**ev.evaluate(y, ev.predict_threshold(s, 0.5), s, key), "decision": "umbral 0,5", **extra})
    rows.append({**ev.evaluate(y, ev.predict_by_rank(s, 0.5), s, key), "decision": "rango 50 %", **extra})

cols = ["modelo", "decision", "f1_fake", "precision", "recall", "pct_pred_fake", "auc", "p_formato", "nan", "s_noticia", "n"]
tabla = ev.results_table(rows)
tabla = tabla[[c for c in cols if c in tabla.columns]]
tabla.to_csv(OUT / "piloto_dev_resumen.csv", index=False)
tabla

,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc,p_formato,nan,s_noticia,n
0,gemma3-12b,rango 50 %,0.880,0.862,0.899,0.500,0.943,1.000,0.0,5.232,290
1,llama31-8b,rango 50 %,0.880,0.862,0.899,0.500,0.944,0.999,0.0,0.853,290
2,llama31-8b,"umbral 0,5",0.856,0.817,0.899,0.528,0.944,0.999,0.0,0.853,290
3,gemma3-12b,"umbral 0,5",0.855,0.940,0.784,0.400,0.943,1.000,0.0,5.232,290
4,qwen3-8b,rango 50 %,0.852,0.834,0.871,0.500,0.918,1.000,0.0,0.839,290
5,char-svm (baseline),umbral,0.806,0.792,0.820,0.497,0.886,NaN,NaN,NaN,290
6,char-svm (baseline),rango 50 %,0.803,0.786,0.820,0.500,0.886,NaN,NaN,NaN,290
7,qwen3-8b,"umbral 0,5",0.789,0.686,0.928,0.648,0.918,1.000,0.0,0.839,290


### Sesgo de cada modelo

Distribución de `p_fake` por clase real. Si las dos clases quedan pegadas a 0 o a 1, el modelo está sesgado hacia una respuesta: el umbral 0,5 falla aunque el AUC sea bueno.

In [16]:
for key, r in piloto.items():
    print(f"== {key}")
    display(r.groupby("label").p_fake.describe(percentiles=[.1, .5, .9]).round(3))

== qwen3-8b


,count,mean,std,min,10%,50%,90%,max
label,,,,,,,,
0,151.0,0.389,0.470,0.0,0.000,0.005,1.0,1.0
1,139.0,0.932,0.244,0.0,0.994,1.000,1.0,1.0


== llama31-8b


,count,mean,std,min,10%,50%,90%,max
label,,,,,,,,
0,151.0,0.191,0.341,0.0,0.000,0.001,0.911,0.999
1,139.0,0.896,0.284,0.0,0.677,0.999,1.000,1.000


== gemma3-12b


,count,mean,std,min,10%,50%,90%,max
label,,,,,,,,
0,151.0,0.047,0.205,0.0,0.0,0.0,0.0,1.0
1,139.0,0.784,0.409,0.0,0.0,1.0,1.0,1.0


## Decisión

*(Rellenar tras ejecutar y pasar al diario.)*

- Modelo elegido:
- AUC / F1 (rango) en dev:
- Motivo:
- Problemas encontrados (formato, `nan`, memoria, tiempo):

In [17]:
!cd /kaggle/working/Practica-TAPLN/results && zip -r /kaggle/working/llm.zip llm


  adding: llm/ (stored 0%)
  adding: llm/piloto_dev_gemma3-12b.csv (deflated 57%)
  adding: llm/piloto_dev_resumen.csv (deflated 58%)
  adding: llm/piloto_dev_llama31-8b.csv (deflated 54%)
  adding: llm/piloto_dev_qwen3-8b.csv (deflated 61%)
